In [ ]:
###Build a Basic Chatbot with langgraph ( Graph API functionality)

In [ ]:
from typing import Annotated

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages # reducers 

In [ ]:
class State(TypedDict): # the values of the state are expected to be in dictionary type
    """
    state for the chatbot : messages
    what type a state key has - here messages is a list
    how that state key shld be updated - via a function like add_messages(append) 
    """
    messages: Annotated[list, add_messages]


graph_builder = StateGraph(State)

In [ ]:
from dotenv import load_dotenv, find_dotenv
import os

print("CWD:", os.getcwd())
print(".env found at:", find_dotenv())        # where python-dotenv thinks .env is
loaded = load_dotenv()
print("load_dotenv() returned:", loaded)      # True if it actually loaded some variables from .env file
print("GROQ_API_KEY value:", os.getenv("GROQ_API_KEY"))

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()


In [ ]:

GROQ_API_KEY = "YOUR KEY"

import os
print(os.getenv("GROQ_API_KEY") is not None)

In [ ]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

llm = ChatGroq(model_name="llama-3.1-8b-instant")

In [ ]:
llm

In [ ]:
# node functionality

def chatbot(state: State):
    # Take the last message as the user input and append the LLM response
    user_input = state["messages"][-1]
    ai_response = llm.invoke(user_input)
    return {"messages": [ai_response]}  # add_messages will append this to the existing list
    # state["messages"] is the list of messages in the state; we use the last item as input to the LLM.

In [ ]:
graph_builder = StateGraph(State)

#adding nodes to the graph 
graph_builder.add_node("llmchatbot", chatbot) # scond parameter is the node functionality 

#adding edges 
graph_builder.add_edge(START, "llmchatbot")
graph_builder.add_edge("llmchatbot", END)

# compile the graph 
graph = graph_builder.compile()

In [ ]:
# to see how the graph looks like 

from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass


In [ ]:
# how to run the graph 

response = graph.invoke({"messages":["Hello, how are you?"]})

In [ ]:
response["messages"]


In [ ]:
# two ways of streaming the response

for event in graph.stream({"messages":["Hello, how are you?"]}):
    #print(event)
    for value in event.values():
        print(value)

In [ ]:
# tools
from langchain_tavily import TavilySearchResults

tool = TavilySearch(max_results=2)
tool.invoke("What is the capital of France?")